# Census Data Engineering Pipeline - Databricks

This notebook implements only the Data Engineering steps `3.1` to `3.8` for the census source file and publishes the curated Delta table to `workshop.default.donation_data_v1`.

In [ ]:
from pyspark.sql import functions as F
from functools import reduce
import re

dbutils.widgets.text("raw_path", "dbfs:/FileStore/tables/census_data_raw.csv")
dbutils.widgets.text("target_table", "workshop.default.donation_data_v1")
dbutils.widgets.text("dq_table", "workshop.default.census_dq_metrics_v1")

raw_path = dbutils.widgets.get("raw_path")
target_table = dbutils.widgets.get("target_table")
dq_table = dbutils.widgets.get("dq_table")

expected_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income", "event_time", "random_flag",
    "source_system"
]

business_key_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "native_country", "event_time_std",
    "income", "source_system"
]


## 3.1 Ingestion Contract

- Read CSV with headers
- Validate schema consistency
- Capture row count and column metadata
- Identify data quality risks: invalid tokens, inconsistent formats, nulls, duplicates

In [ ]:
raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(raw_path)
)

raw_row_count = raw_df.count()
raw_column_metadata = [(field.name, field.dataType.simpleString(), field.nullable) for field in raw_df.schema.fields]

display(spark.createDataFrame(raw_column_metadata, ["column_name", "data_type", "nullable"]))
print(f"Raw row count: {raw_row_count}")


In [ ]:
def standardize_column_name(name: str) -> str:
    value = re.sub(r"[^0-9a-zA-Z]+", "_", name.strip().lower())
    value = re.sub(r"_+", "_", value).strip("_")
    return value

raw_standardized_columns = [standardize_column_name(c) for c in raw_df.columns]
missing_columns = sorted(set(expected_columns) - set(raw_standardized_columns))
extra_columns = sorted(set(raw_standardized_columns) - set(expected_columns))

if missing_columns:
    raise ValueError(f"Schema validation failed. Missing columns: {missing_columns}")

schema_contract_df = spark.createDataFrame(
    [(raw_row_count, len(raw_df.columns), missing_columns, extra_columns)],
    ["row_count", "column_count", "missing_columns", "extra_columns"]
)
display(schema_contract_df)


In [ ]:
raw_null_exprs = [F.sum(F.when(F.trim(F.col(c).cast("string")).isin("", "null", "NULL", "None"), 1).when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in raw_df.columns]
raw_null_counts_df = raw_df.select(raw_null_exprs)

invalid_token_conditions = [F.lower(F.trim(F.col(c).cast("string"))).rlike("^error_?") for c in raw_df.columns]
invalid_token_count = raw_df.filter(reduce(lambda left, right: left | right, invalid_token_conditions)).count()
raw_duplicate_count = raw_row_count - raw_df.dropDuplicates().count()

display(raw_null_counts_df)
display(spark.createDataFrame([(invalid_token_count, raw_duplicate_count)], ["invalid_token_count", "raw_duplicate_count"]))


## 3.2 Column Standardization

- Convert all column names to lowercase
- Trim whitespace
- Ensure deterministic schema ordering

In [ ]:
df = raw_df.toDF(*raw_standardized_columns)
df = df.select(*expected_columns)

display(spark.createDataFrame([(df.columns,)], ["deterministic_column_order"]))
df.printSchema()


## 3.3 Data Cleaning Rules

- Age: remove `ERROR_*` tokens, strip non-digits, cast to numeric
- Workclass: replace nulls with `Unknown`
- Trim all categorical columns
- Remove duplicates based on business keys

In [ ]:
def trim_to_null(column_name: str):
    value = F.trim(F.col(column_name).cast("string"))
    return F.when(value.isNull() | (value == "") | F.lower(value).isin("null", "none", "nan"), None).otherwise(value)

def clean_age(column_name: str):
    value = F.lower(F.trim(F.col(column_name).cast("string")))
    digits = F.regexp_extract(value, "([0-9]+)", 1)
    return F.when(value.isNull() | (value == "") | value.rlike("^error_?") | (digits == ""), None).otherwise(digits.cast("int"))

categorical_columns = [
    "workclass", "education_level", "marital_status", "occupation", "relationship",
    "race", "sex", "native_country", "income", "random_flag", "source_system"
]

clean_df = df
for column_name in df.columns:
    clean_df = clean_df.withColumn(column_name, trim_to_null(column_name))

for column_name in categorical_columns:
    clean_df = clean_df.withColumn(column_name, trim_to_null(column_name))

clean_df = (
    clean_df
    .withColumn("age_raw", F.col("age"))
    .withColumn("age", clean_age("age"))
    .withColumn("workclass", F.coalesce(F.col("workclass"), F.lit("Unknown")))
)

display(clean_df.limit(20))


## 3.4 Event Time Parsing

- Parse multiple timestamp formats into `event_time_std`
- Maintain parsing order: ISO, epoch, custom formats
- Track parse failure counts

In [ ]:
event_text = F.trim(F.col("event_time").cast("string"))

clean_df = clean_df.withColumn(
    "event_time_std",
    F.coalesce(
        F.expr("try_to_timestamp(event_time)"),
        F.when(event_text.rlike("^[0-9]{10}$"), F.to_timestamp(F.from_unixtime(event_text.cast("bigint")))),
        F.when(event_text.rlike("^[0-9]{13}$"), F.to_timestamp(F.from_unixtime((event_text.cast("double") / F.lit(1000)).cast("bigint")))),
        F.expr("try_to_timestamp(event_time, 'dd/MM/yy HH:mm')"),
        F.expr("try_to_timestamp(event_time, 'dd/MM/yy')"),
        F.expr("try_to_timestamp(event_time, 'MM/dd/yy HH:mm')"),
        F.expr("try_to_timestamp(event_time, 'MM/dd/yy')")
    )
)

timestamp_parse_failure_count = clean_df.filter(F.col("event_time").isNotNull() & F.col("event_time_std").isNull()).count()

display(spark.createDataFrame([(timestamp_parse_failure_count,)], ["timestamp_parse_failure_count"]))
display(clean_df.select("event_time", "event_time_std").limit(20))


## 3.5 Label Normalization

- Normalize income into two values: `<=80k` and `>80k`
- Map all variations to canonical labels
- Validate no drift in label vocabulary

In [ ]:
income_normalized = F.lower(F.regexp_replace(F.trim(F.col("income").cast("string")), "\\s+", ""))

clean_df = clean_df.withColumn("income_raw", F.col("income"))
clean_df = clean_df.withColumn(
    "income",
    F.when(income_normalized.isin("<=80k", "=<80k", "le80k", "lessorequal80k", "low", "0"), "<=80k")
    .when(income_normalized.isin(">80k", "gt80k", "greaterthan80k", "high", "1"), ">80k")
    .otherwise(None)
)

allowed_income_labels = ["<=80k", ">80k"]
label_drift_count = clean_df.filter(F.col("income").isNotNull() & ~F.col("income").isin(allowed_income_labels)).count()
label_normalization_failure_count = clean_df.filter(F.col("income_raw").isNotNull() & F.col("income").isNull()).count()

if label_drift_count > 0:
    raise ValueError(f"Income label drift found: {label_drift_count}")

display(clean_df.groupBy("income").count())
display(spark.createDataFrame([(label_drift_count, label_normalization_failure_count)], ["label_drift_count", "label_normalization_failure_count"]))


## 3.6 Numeric Casting

- Clean numeric columns: remove `$`, `%`, commas, and `k` suffix
- Cast to double
- Validate no cast failures

In [ ]:
def clean_numeric(column_name: str):
    value = F.lower(F.trim(F.col(column_name).cast("string")))
    without_commas = F.regexp_replace(value, ",", "")
    multiplier = F.when(without_commas.rlike("k$"), F.lit(1000.0)).otherwise(F.lit(1.0))
    numeric_text = F.regexp_replace(without_commas, "[^0-9.-]", "")
    is_valid_number = numeric_text.rlike("^-?[0-9]+(\\.[0-9]+)?$")
    return F.when(value.isNull() | (value == "") | value.rlike("^error_?") | ~is_valid_number, None).otherwise(numeric_text.cast("double") * multiplier)

numeric_columns = ["education_num", "capital_gain", "capital_loss", "hours_per_week"]

for column_name in numeric_columns:
    clean_df = clean_df.withColumn(f"{column_name}_raw", F.col(column_name))
    clean_df = clean_df.withColumn(column_name, clean_numeric(column_name))

numeric_cast_failure_rows = []
for column_name in numeric_columns:
    raw_value = F.lower(F.trim(F.col(f"{column_name}_raw").cast("string")))
    failure_count = clean_df.filter(raw_value.isNotNull() & (raw_value != "") & ~raw_value.rlike("^error_?") & F.col(column_name).isNull()).count()
    numeric_cast_failure_rows.append((column_name, failure_count))

numeric_cast_failure_df = spark.createDataFrame(numeric_cast_failure_rows, ["column_name", "cast_failure_count"])
numeric_cast_failure_count = numeric_cast_failure_df.agg(F.sum("cast_failure_count").alias("total_failures")).collect()[0]["total_failures"] or 0

if numeric_cast_failure_count > 0:
    raise ValueError(f"Numeric cast failures found: {numeric_cast_failure_count}")

display(numeric_cast_failure_df)
display(clean_df.select("education_num", "capital_gain", "capital_loss", "hours_per_week").summary())


### Business Key De-duplication

This completes the duplicate-removal rule from `3.3`. It runs after timestamp, label, and numeric standardization so the business key uses clean values.

In [ ]:
before_dedup_count = clean_df.count()
dedup_df = clean_df.dropDuplicates(business_key_columns)
after_dedup_count = dedup_df.count()
business_key_duplicate_count = before_dedup_count - after_dedup_count

display(spark.createDataFrame([(before_dedup_count, after_dedup_count, business_key_duplicate_count)], ["before_dedup_count", "after_dedup_count", "business_key_duplicate_count"]))


## 3.7 Data Quality Metrics

- Null counts
- Invalid record counts
- Timestamp parse failures
- Label consistency checks

In [ ]:
publish_columns = [
    "age", "workclass", "education_level", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income", "event_time_std", "random_flag",
    "source_system"
]

curated_df = (
    dedup_df.select(*publish_columns)
    .withColumn("processed_at", F.current_timestamp())
)

invalid_age_count = curated_df.filter((F.col("age") < 0) | (F.col("age") > 120)).count()
invalid_hours_count = curated_df.filter((F.col("hours_per_week") < 0) | (F.col("hours_per_week") > 168)).count()
invalid_record_count = invalid_token_count + invalid_age_count + invalid_hours_count + label_normalization_failure_count + timestamp_parse_failure_count

metric_rows = [
    ("raw_row_count", raw_row_count),
    ("curated_row_count", curated_df.count()),
    ("raw_duplicate_count", raw_duplicate_count),
    ("business_key_duplicate_count", business_key_duplicate_count),
    ("invalid_token_count", invalid_token_count),
    ("invalid_age_count", invalid_age_count),
    ("invalid_hours_per_week_count", invalid_hours_count),
    ("invalid_record_count", invalid_record_count),
    ("timestamp_parse_failure_count", timestamp_parse_failure_count),
    ("label_drift_count", label_drift_count),
    ("label_normalization_failure_count", label_normalization_failure_count),
    ("numeric_cast_failure_count", numeric_cast_failure_count),
]

for column_name in curated_df.columns:
    null_count = curated_df.filter(F.col(column_name).isNull()).count()
    metric_rows.append((f"null_count__{column_name}", null_count))

dq_metrics_df = (
    spark.createDataFrame(metric_rows, ["metric_name", "metric_value"])
    .withColumn("metric_value", F.col("metric_value").cast("long"))
    .withColumn("source_path", F.lit(raw_path))
    .withColumn("measured_at", F.current_timestamp())
)

display(dq_metrics_df.orderBy("metric_name"))


## 3.8 Publish Layer

- Write to Delta table: `workshop.default.donation_data_v1`
- Use overwrite mode for idempotency
- Validate schema and row count

In [ ]:
def ensure_table_namespace(table_name: str):
    parts = table_name.split(".")
    if len(parts) == 3:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {parts[0]}.{parts[1]}")
    elif len(parts) == 2:
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {parts[0]}")

ensure_table_namespace(target_table)
ensure_table_namespace(dq_table)

(
    curated_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

(
    dq_metrics_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dq_table)
)


In [ ]:
published_df = spark.table(target_table)
published_count = published_df.count()
curated_count = curated_df.count()

if published_count != curated_count:
    raise ValueError(f"Row count validation failed: published={published_count}, curated={curated_count}")

published_columns = published_df.columns
expected_publish_columns = curated_df.columns

if published_columns != expected_publish_columns:
    raise ValueError(f"Schema ordering validation failed: {published_columns} != {expected_publish_columns}")

display(published_df.limit(20))
print(f"Published Delta table: {target_table}")
print(f"Published DQ metrics table: {dq_table}")
print(f"Validated row count: {published_count}")
